In [ ]:
# === Notebook common preamble (load the llm_math package) ===
import sys
from pathlib import Path

# Candidate paths for the llm_math package
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# Add parent directories as candidates (when running from the notebooks folder)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# Try importing llm_math
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[Warning] load the llm_math package text: {e}")
    print("  Clone the GitHub repository and run colab_setup.sh.")
# === end preamble ===


# Ch 11. RNNtext LSTM — text text text text

> **Learning Goals**
> - RNNtext text text text text
> - text text(long-term dependency) Problemtext text text text
> - LSTMtext text text text text text text

## 11.1 text Datatext text text Problem

text text text Vectortext, text text text text. text text?

- **RNN**: text text text text
- **CNN (1D)**: text text text
- **Transformer (Ch 14+)**: Attentiontext text text

text text RNN/LSTMtext text. text text text.

## 11.2 RNNtext text text

$$\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t + \mathbf{b})$$

- $\mathbf{x}_t \in \mathbb{R}^d$: Time $t$text text
- $\mathbf{h}_t \in \mathbb{R}^h$: Time $t$text text text (text)
- $W_h \in \mathbb{R}^{h \times h}$, $W_x \in \mathbb{R}^{h \times d}$: text

text: **text text text Time text text** (text text).


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# text RNN text (NumPy)
class RNNCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        self.Wx = rng.standard_normal((input_dim, hidden_dim)) * 0.1
        self.Wh = rng.standard_normal((hidden_dim, hidden_dim)) * 0.1
        self.b = np.zeros(hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        """x_seq: (T, input_dim) -> h_seq: (T, hidden_dim)"""
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        h_seq = []
        for t in range(T):
            h = np.tanh(x_seq[t] @ self.Wx + h @ self.Wh + self.b)
            h_seq.append(h.copy())
        return np.array(h_seq)

    def last_hidden(self, x_seq):
        return self.forward(x_seq)[-1]

# Test
rnn = RNNCell(input_dim=4, hidden_dim=8)
x_seq = np.random.randn(10, 4)  # Length 10 text
h_seq = rnn.forward(x_seq)
print(f"text text: {x_seq.shape}")
print(f"text text text: {h_seq.shape}")
print(f"text text text: {h_seq[-1].round(4)}")


## 11.3 BPTT (Backpropagation Through Time)

RNNtext Backpropagation = Time text text Calculation Graphtext Backpropagation.

$$\frac{\partial \mathcal{L}}{\partial \mathbf{h}_0} = \prod_{t=1}^{T} \frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} \cdot \frac{\partial \mathcal{L}}{\partial \mathbf{h}_T}$$

$\frac{\partial \mathbf{h}_t}{\partial \mathbf{h}_{t-1}} = \mathrm{diag}(1 - \mathbf{h}_t^2) W_h$ (tanhtext Derivative text).

Problem: $\mathrm{diag}(1 - \mathbf{h}_t^2) \leq 1$text $W_h$text textValue textValuetext 1text text **text text**, text **text**.


In [ ]:
# text text/text text
def simulate_rnn_gradient(T, Wh_norm, seed=0):
    """T Step RNNtext h_0text text Gradient Magnitude."""
    rng = np.random.default_rng(seed)
    h = rng.standard_normal(8)
    grad = np.ones(8)  # dL/dh_T
    for t in range(T):
        # Whtext text Matrixtext normtext
        Wh = np.eye(8) * Wh_norm
        h_prev = rng.standard_normal(8)
        h = np.tanh(h_prev @ Wh)
        # Backward Pass: grad = grad @ Wh @ diag(1 - h^2)
        diag = 1 - h**2
        grad = grad @ Wh * diag
    return np.linalg.norm(grad)

print("RNN text text/text text:")
print(f"{'T':>5} | {'||Wh||=0.5 (text)':>20} | {'||Wh||=1.0':>15} | {'||Wh||=1.5 (text)':>20}")
print("-" * 65)
for T in [5, 10, 20, 50, 100]:
    g_small = simulate_rnn_gradient(T, 0.5)
    g_one = simulate_rnn_gradient(T, 1.0)
    g_big = simulate_rnn_gradient(T, 1.5)
    print(f"{T:>5} | {g_small:>20.4e} | {g_one:>15.4e} | {g_big:>20.4e}")

print("\n=> ||Wh||<1textPlane Gradienttext text Decrease (text)")
print("=> ||Wh||>1textPlane Gradienttext text Increase (text)")


## 11.4 LSTM — text text

LSTM (1997, Hochreiter & Schmidhuber)text text text text text.

$$\mathbf{f}_t = \sigma(W_f [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_f) \quad \text{(forget gate)}$$
$$\mathbf{i}_t = \sigma(W_i [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_i) \quad \text{(input gate)}$$
$$\tilde{\mathbf{c}}_t = \tanh(W_c [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_c) \quad \text{(candidate)}$$
$$\mathbf{c}_t = \mathbf{f}_t \odot \mathbf{c}_{t-1} + \mathbf{i}_t \odot \tilde{\mathbf{c}}_t \quad \text{(cell state)}$$
$$\mathbf{o}_t = \sigma(W_o [\mathbf{h}_{t-1}, \mathbf{x}_t] + \mathbf{b}_o) \quad \text{(output gate)}$$
$$\mathbf{h}_t = \mathbf{o}_t \odot \tanh(\mathbf{c}_t) \quad \text{(hidden state)}$$

text: **cell state $\mathbf{c}_t$text text text** → text text text.


In [ ]:
# LSTM text text (NumPy)
class LSTMCell:
    def __init__(self, input_dim, hidden_dim, seed=0):
        rng = np.random.default_rng(seed)
        # 4text text: forget, input, candidate, output
        self.W = rng.standard_normal((input_dim + hidden_dim, 4 * hidden_dim)) * 0.1
        self.b = np.zeros(4 * hidden_dim)
        self.hidden_dim = hidden_dim

    def forward(self, x_seq):
        T = x_seq.shape[0]
        h = np.zeros(self.hidden_dim)
        c = np.zeros(self.hidden_dim)
        h_seq, c_seq = [], []
        for t in range(T):
            # text: [h, x]
            hx = np.concatenate([h, x_seq[t]])
            gates = hx @ self.W + self.b
            f, i, g, o = np.split(gates, 4)
            f = 1 / (1 + np.exp(-f))  # sigmoid
            i = 1 / (1 + np.exp(-i))
            g = np.tanh(g)
            o = 1 / (1 + np.exp(-o))
            c = f * c + i * g  # cell state (text text!)
            h = o * np.tanh(c)
            h_seq.append(h.copy()); c_seq.append(c.copy())
        return np.array(h_seq), np.array(c_seq)

# text
lstm = LSTMCell(input_dim=4, hidden_dim=8)
h_seq, c_seq = lstm.forward(x_seq)
print(f"LSTM Output: h_seq {h_seq.shape}, c_seq {c_seq.shape}")
print(f"text h: {h_seq[-1].round(4)}")
print(f"text c: {c_seq[-1].round(4)}")


## 11.5 text text text Model

text text text RNN/LSTMtext text text text.


In [ ]:
# text text LSTM text Model (PyTorch)
import torch
import torch.nn as nn

# Tiny Shakespeare text text
from llm_math.data import load_tiny_shakespeare
text = load_tiny_shakespeare(max_chars=20000)
chars = sorted(set(text))
char_to_idx = {c: i for i, c in enumerate(chars)}
idx_to_char = {i: c for i, c in enumerate(chars)}
vocab_size = len(chars)
print(f"text Length: {len(text)}, Vocabulary Size: {vocab_size}")
print(f"text: {chars[:20]}...")

# Data text
seq_len = 50
data = np.array([char_to_idx[c] for c in text])

def make_batch(data, seq_len, batch_size=32):
    """text Batch Generation."""
    n = len(data) - seq_len - 1
    starts = np.random.randint(0, n, batch_size)
    X = np.array([data[s:s+seq_len] for s in starts])
    y = np.array([data[s+1:s+1+seq_len] for s in starts])
    return torch.tensor(X, dtype=torch.long), torch.tensor(y, dtype=torch.long)

# Model
class CharLSTM(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden_dim=128, n_layers=1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim)
        self.lstm = nn.LSTM(embed_dim, hidden_dim, n_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x, hidden=None):
        emb = self.embed(x)  # (B, T, E)
        out, hidden = self.lstm(emb, hidden)  # (B, T, H)
        logits = self.fc(out)  # (B, T, V)
        return logits, hidden

model = CharLSTM(vocab_size, embed_dim=32, hidden_dim=64)
n_params = sum(p.numel() for p in model.parameters())
print(f"\nModel text text: {n_params:,}")


In [ ]:
# Training
import time

loss_fn = nn.CrossEntropyLoss()
opt = torch.optim.Adam(model.parameters(), lr=0.003)

n_steps = 300
losses = []
t0 = time.time()
for step in range(n_steps):
    X, y = make_batch(data, seq_len, batch_size=32)
    opt.zero_grad()
    logits, _ = model(X)
    # (B*T, V) vs (B*T,)
    loss = loss_fn(logits.reshape(-1, vocab_size), y.reshape(-1))
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    opt.step()
    losses.append(loss.item())
    if (step + 1) % 50 == 0:
        print(f"Step {step+1}: loss={loss.item():.4f}")
print(f"\nTraining Time: {time.time() - t0:.1f}text")


In [ ]:
# text text
def generate(model, seed_text, length=200, temperature=0.8):
    model.eval()
    chars_idx = [char_to_idx[c] for c in seed_text]
    hidden = None
    with torch.no_grad():
        x = torch.tensor([chars_idx], dtype=torch.long)
        for _ in range(length):
            logits, hidden = model(x, hidden)
            # text Time text
            logit = logits[0, -1] / temperature
            probs = torch.softmax(logit, dim=0)
            next_idx = torch.multinomial(probs, 1).item()
            chars_idx.append(next_idx)
            # text text text text
            x = torch.tensor([[next_idx]], dtype=torch.long)
    return ''.join(idx_to_char[i] for i in chars_idx)

print("=== Generationtext text (Training text) ===\n")
print(generate(model, "First Citizen:\n", length=300))


In [ ]:
# Training text
plt.figure(figsize=(10, 4))
plt.plot(losses, alpha=0.3, color='blue')
# text Mean
window = 20
smoothed = np.convolve(losses, np.ones(window)/window, mode='valid')
plt.plot(range(window-1, len(losses)), smoothed, 'r-', linewidth=2, label='textMean')
plt.xlabel('Step'); plt.ylabel('Loss')
plt.title('Character-level LSTM Learning Curve')
plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../figures/ch11_char_lstm.png', dpi=100, bbox_inches='tight')
plt.show()


## 11.6 GRU — LSTMtext text

GRU (2014, Cho et al.)text text 2text text:

$$\mathbf{z}_t = \sigma(W_z [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(update gate)}$$
$$\mathbf{r}_t = \sigma(W_r [\mathbf{h}_{t-1}, \mathbf{x}_t]) \quad \text{(reset gate)}$$
$$\tilde{\mathbf{h}}_t = \tanh(W [\mathbf{r}_t \odot \mathbf{h}_{t-1}, \mathbf{x}_t])$$
$$\mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \tilde{\mathbf{h}}_t$$

cell statetext text text text. Performancetext LSTMtext text.


## 11.7 Attentiontext text text — RNNtext text

RNNtext text:
1. **text text**: text text → text
2. **text text**: LSTMdegrees text text text
3. **text text text text**: text text text text Vectortext text

**Attention(attention)**text text text text text text text text text text text text text. (Bahdanau et al., 2014)

text text(Ch 14)text text text.

## 11.8 Key Takeaways

| text | text | text |
|---|---|---|
| RNN | $\mathbf{h}_t = \tanh(W_h \mathbf{h}_{t-1} + W_x \mathbf{x}_t)$ | text, text/text |
| LSTM | text + cell state (text) | text text text |
| GRU | text 2text (text) | LSTMtext text |
| BPTT | Timetext Backpropagation | $\prod \frac{\partial h_t}{\partial h_{t-1}}$ |

## Exercises

1. RNNtext "hello world"text text text Trainingtext text.
2. LSTMtext GRUtext text Datatext Trainingtext text Speedtext Comparisontext.
3. Sequence Length 10, 50, 100text text RNNtext LSTMtext text text Comparisontext.
4. RNNtext text text text text text text text text.
5. text text LSTMtext temperaturetext 0.3, 0.8, 1.5text text text text Comparisontext.

> Solutions: `solutions/ch11_solutions.ipynb`
